In [2]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments, Trainer
)
import numpy as np

# 1. Load the dataset and select the three languages
raw_datasets = load_dataset("coastalcph/tydi_xor_rc")
languages = ["ko", "te", "ar"]

train_datasets = {
    lang: raw_datasets["train"].filter(lambda x, l=lang: x["lang"] == l)
    for lang in languages
}
val_datasets = {
    lang: raw_datasets["validation"].filter(lambda x, l=lang: x["lang"] == l)
    for lang in languages
}

# 2. Choose a multilingual model
checkpoint = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def preprocess_function(examples):
    """Convert each (question, context) pair into input_ids, attention_mask and per-token labels."""
    questions = [q.strip() for q in examples["question"]]
    # Tokenize and get offset mappings
    encoding = tokenizer(
        questions,
        examples["context"],
        max_length=512,
        truncation="only_second",
        return_offsets_mapping=True,
        padding="max_length",
    )
    offset_mapping = encoding.pop("offset_mapping")
    labels = []

    for i, offsets in enumerate(offset_mapping):
        # Determine the answer span; if unanswerable, start_char/end_char remain None
        if not examples["answerable"][i] or examples["answer_start"][i] == -1:
            start_char = end_char = None
        else:
            start_char = examples["answer_start"][i]
            end_char   = start_char + len(examples["answer"][i])

        sequence_ids = encoding.sequence_ids(i)
        label = []
        for seq_id, (start, end) in zip(sequence_ids, offsets):
            if seq_id != 1:
                # token belongs to the question or a special token → ignore in loss
                label.append(-100)
            else:
                if start_char is None:
                    # unanswerable → all context tokens are 0
                    label.append(0)
                else:
                    # mark tokens whose span overlaps the answer as 1, else 0
                    is_answer = (start < end_char) and (end > start_char)
                    label.append(1 if is_answer else 0)
        labels.append(label)

    encoding["labels"] = labels
    return encoding

def make_compute_metrics(answerable_flags):
    """
    Returns a compute_metrics function for HF Trainer that:
      - Keeps token-level precision/recall/F1/accuracy
      - Adds token-acc on answerable/unanswerable subsets
      - Adds question-level accuracy on answerable/unanswerable
    `answerable_flags` must be a 1D bool/0-1 array aligned with eval_dataset order.
    """
    ans_flags = np.asarray(answerable_flags, dtype=bool)

    def compute_metrics(eval_pred):
        predictions, labels = eval_pred  # predictions: [N, L, C], labels: [N, L]
        if isinstance(predictions, tuple):  # some HF versions return (logits, ...)
            predictions = predictions[0]
        preds = np.argmax(predictions, axis=-1)  # [N, L]

        # --- Global token-level confusion counts (ignoring -100) ---
        TP = FP = FN = TN = 0
        token_acc_per_example = []   # per-example token accuracy
        q_correct_ans = []           # question-level correct for answerable examples
        q_correct_un  = []           # question-level correct for unanswerable examples

        N = labels.shape[0]
        for i in range(N):
            mask = (labels[i] != -100)          # only context tokens (you set others to -100)
            if not np.any(mask):
                # edge case: no valid tokens; skip from per-example stats
                continue

            true_i = labels[i][mask]            # 0/1
            pred_i = preds[i][mask]             # 0/1

            TP += np.sum((true_i == 1) & (pred_i == 1))
            FP += np.sum((true_i == 0) & (pred_i == 1))
            FN += np.sum((true_i == 1) & (pred_i == 0))
            TN += np.sum((true_i == 0) & (pred_i == 0))

            # Per-example token accuracy
            token_acc_per_example.append(np.mean(true_i == pred_i))

            # Question-level correctness
            if ans_flags[i]:
                # Correct if predicted at least one of the true answer tokens
                # (overlap of predicted=1 with gold=1)
                q_correct_ans.append(bool(np.any((true_i == 1) & (pred_i == 1))))
            else:
                # Correct if predicted no answer tokens at all
                q_correct_un.append(not bool(np.any(pred_i == 1)))

        precision = TP / (TP + FP + 1e-8)
        recall    = TP / (TP + FN + 1e-8)
        f1        = 2 * precision * recall / (precision + recall + 1e-8)
        accuracy  = (TP + TN) / (TP + TN + FP + FN + 1e-8)

        token_acc_per_example = np.array(token_acc_per_example, dtype=float)

        # Build masks over examples (length N) for token-acc splits
        mask_ans = ans_flags[:N]
        mask_un  = ~ans_flags[:N]

        # To align with token_acc_per_example length, we recompute per-example token acc only
        # for examples that had any valid tokens; build a parallel list of indices we counted:
        counted_idx = []
        for i in range(N):
            if np.any(labels[i] != -100):
                counted_idx.append(i)
        counted_idx = np.array(counted_idx, dtype=int)

        # Split token accuracy by answerable/unanswerable on counted examples
        if counted_idx.size > 0:
            ans_mask_counted = mask_ans[counted_idx]
            un_mask_counted  = mask_un[counted_idx]
            token_acc_ans = float(token_acc_per_example[ans_mask_counted].mean()) if np.any(ans_mask_counted) else float("nan")
            token_acc_un  = float(token_acc_per_example[un_mask_counted].mean())  if np.any(un_mask_counted)  else float("nan")
        else:
            token_acc_ans = token_acc_un = float("nan")

        # Question-level accuracies
        q_acc_ans = float(np.mean(q_correct_ans)) if len(q_correct_ans) > 0 else float("nan")
        q_acc_un  = float(np.mean(q_correct_un))  if len(q_correct_un)  > 0 else float("nan")

        return {
            # Original token-level metrics
            "precision": precision,
            "recall":    recall,
            "f1":        f1,
            "accuracy":  accuracy,
            # New: token-accuracy by question type
            "token_acc_answerable":   token_acc_ans,
            "token_acc_unanswerable": token_acc_un,
            # New: question-level accuracy by question type
            "q_acc_answerable":   q_acc_ans,
            "q_acc_unanswerable": q_acc_un,
        }

    return compute_metrics

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

results = {}
for lang in languages:
    # 3. Preprocess the data
    tokenized_train = train_datasets[lang].map(
        preprocess_function,
        batched=True,
        remove_columns=train_datasets[lang].column_names,
    )
    tokenized_val = val_datasets[lang].map(
        preprocess_function,
        batched=True,
        remove_columns=val_datasets[lang].column_names,
    )

    # 4. Load a fresh model for this language
    model = AutoModelForTokenClassification.from_pretrained(checkpoint, num_labels=2)

    # 5. Set up training arguments; adjust as needed for your resources
    training_args = TrainingArguments(
        output_dir=f"tydi_{lang}_token_classifier",
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=2,
        weight_decay=0.01,
        report_to = []
    )

    # 6. Train and evaluate
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=make_compute_metrics(np.array(val_datasets[lang]["answerable"], dtype=bool)),
    )

    trainer.train()
    metrics = trainer.evaluate()
    results[lang] = metrics
    print(f"Validation metrics for {lang}: {metrics}")

print("Final results:", results)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Map:   0%|          | 0/2422 [00:00<?, ? examples/s]

Map:   0%|          | 0/356 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2038738711.py:198: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 